In [10]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

TF version: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 0 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
GPU 1 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}


# Abordagem 1: Detecção de outliers (Clustering)

In [ ]:
# Experimento E4: One-Class SVM (tabular) — esqueleto
# Ideia: treinar só na classe "normal" (ex.: sMCI) e usar o score como anomalia.
# Avaliação: AUC para detectar pMCI como "anomalia" (pMCI=1).

from sklearn.svm import OneClassSVM

NORMAL_LABEL = "sMCI"  # escolha: "sMCI" como normal e "pMCI" como anomalia
nu = 0.1
kernel = "rbf"
gamma = "scale"

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=True)  # geralmente melhor após seleção
    X_tr, y_tr = split_xy(tr)
    X_va, y_va = split_xy(va)

    # garante mesmas colunas
    common = [c for c in X_tr.columns if c in X_va.columns]
    X_tr = X_tr[common]
    X_va = X_va[common]

    # treina apenas no "normal" (0 = sMCI no split_xy)
    normal_code = 0 if NORMAL_LABEL == "sMCI" else 1
    X_tr_norm = X_tr[y_tr == normal_code]

    if X_tr_norm.shape[0] < 10:
        raise SystemExit(f"Fold {fold}: poucas amostras normais para OneClassSVM: {X_tr_norm.shape[0]}")

    oc = OneClassSVM(nu=nu, kernel=kernel, gamma=gamma)
    oc.fit(X_tr_norm)

    # score_samples: maior = mais inlier; para anomalia usamos o negativo
    inlier_score = oc.score_samples(X_va)
    anomaly_score = -inlier_score

    # y_va já é 0/1 (0=sMCI,1=pMCI). Se normal for pMCI, inverta o alvo.
    y_anom = y_va if NORMAL_LABEL == "sMCI" else (1 - y_va)

    auc = float(roc_auc_score(y_anom, anomaly_score))
    rows.append(
        {
            "fold": fold,
            "model": "OneClassSVM",
            "normal": NORMAL_LABEL,
            "auc_anomaly": auc,
            "n_features": int(X_tr.shape[1]),
            "n_train_normal": int(X_tr_norm.shape[0]),
            "nu": float(nu),
            "kernel": kernel,
        }
    )

pd.DataFrame(rows)


In [ ]:
# One-Class Graph SVM — esqueleto (Graph Kernel + OneClassSVM)
# Estratégia padrão: representar cada amostra como um grafo e usar um kernel de grafos.
# Depois, treinar OneClassSVM com kernel='precomputed' no conjunto de grafos "normais".
#
# OBS: isso normalmente usa a lib GraKeL. Se não estiver instalada, o bloco não roda,
# mas o esqueleto fica pronto.

NORMAL_LABEL = "sMCI"  # classe considerada "normal"

try:
    import networkx as nx
    from grakel import Graph
    from grakel.kernels import WeisfeilerLehman, VertexHistogram
except Exception as e:
    raise SystemExit(
        "Dependências para Graph SVM ausentes. Instale 'grakel' e 'networkx' para rodar este experimento."
    ) from e

from sklearn.svm import OneClassSVM


def build_nx_graph_from_sample(df_sample: pd.DataFrame) -> nx.Graph:
    """Converte uma amostra (linhas ROI/pair) em um grafo.

    Este é um ESQUELETO: você precisa definir:
    - quais são os nós (ex.: 20 ROIs x side)
    - como ligar as arestas (ex.: fully-connected, atlas, kNN por distância)
    - quais features vão para cada nó

    Sugestão rápida: agregar por (roi,side) e usar a média das colunas numéricas como atributo do nó.
    """
    g = nx.Graph()

    # 1) agrega por nó
    keys = [k for k in ["roi", "side"] if k in df_sample.columns]
    if not keys:
        raise ValueError("A amostra não tem colunas 'roi'/'side' para definir nós.")

    X = df_sample.drop(columns=[c for c in NON_FEATURE_COLS if c in df_sample.columns], errors="ignore")
    X = X.apply(pd.to_numeric, errors="coerce")

    agg = pd.concat([df_sample[keys], X], axis=1).groupby(keys, as_index=False).mean(numeric_only=True)

    # 2) adiciona nós com atributos
    node_ids = []
    for _, row in agg.iterrows():
        node = f"{row['roi']}_{row['side']}" if 'side' in agg.columns else str(row['roi'])
        node_ids.append(node)
        attrs = row.drop(labels=keys).to_dict()
        # GraKeL usa rótulos/atributos; aqui colocamos atributos numéricos e também um label fixo.
        g.add_node(node, attr=attrs)

    # 3) arestas (esqueleto: fully-connected)
    for i in range(len(node_ids)):
        for j in range(i + 1, len(node_ids)):
            g.add_edge(node_ids[i], node_ids[j])

    return g


def nx_to_grakel(g: nx.Graph) -> Graph:
    # GraKeL Graph: (adjacency, node_labels)
    # Para simplificar o esqueleto, usamos um label constante por nó.
    nodes = list(g.nodes())
    idx = {n: i for i, n in enumerate(nodes)}

    adj = {idx[u]: [idx[v] for v in g.neighbors(u)] for u in nodes}
    labels = {idx[u]: 1 for u in nodes}  # rótulos discretos (necessário p/ WL+VH)
    return Graph(adj, node_labels=labels)


# Kernel: Weisfeiler-Lehman + VertexHistogram (bom baseline)
KERNEL = WeisfeilerLehman(n_iter=3, base_graph_kernel=VertexHistogram, normalize=True)

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=False)

    # 1) transforma cada conjunto (ID_PT,COMBINATION_NUMBER,TRIPLET_IDX) em um grafo
    set_keys = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]
    tr_sets = tr[set_keys + ["GROUP"]].drop_duplicates(subset=set_keys)
    va_sets = va[set_keys + ["GROUP"]].drop_duplicates(subset=set_keys)

    def build_graphs(df_all: pd.DataFrame, df_sets: pd.DataFrame):
        Gs = []
        ys = []
        for _, r in df_sets.iterrows():
            mask = (df_all["ID_PT"] == r["ID_PT"]) & (df_all["COMBINATION_NUMBER"] == r["COMBINATION_NUMBER"]) & (df_all["TRIPLET_IDX"] == r["TRIPLET_IDX"])
            sample = df_all.loc[mask]
            g_nx = build_nx_graph_from_sample(sample)
            Gs.append(nx_to_grakel(g_nx))
            ys.append(str(r["GROUP"]).strip())
        return Gs, np.array(ys)

    G_tr, y_tr_s = build_graphs(tr, tr_sets)
    G_va, y_va_s = build_graphs(va, va_sets)

    # 2) kernel matrices
    K_tr = KERNEL.fit_transform(G_tr)          # (n_tr, n_tr)
    K_va = KERNEL.transform(G_va)              # (n_va, n_tr)

    # 3) one-class: treina só nos normais
    y_tr_norm = (y_tr_s == NORMAL_LABEL)
    if y_tr_norm.sum() < 5:
        raise SystemExit(f"Fold {fold}: poucas amostras normais para OneClass Graph SVM: {int(y_tr_norm.sum())}")

    oc = OneClassSVM(kernel="precomputed", nu=0.1)
    oc.fit(K_tr[np.ix_(y_tr_norm, y_tr_norm)])

    # 4) score no val: precisamos do kernel (val x train_normal)
    K_va_norm = K_va[:, y_tr_norm]
    inlier_score = oc.score_samples(K_va_norm)
    anomaly_score = -inlier_score

    y_va_anom = (y_va_s != NORMAL_LABEL).astype(int)
    auc = float(roc_auc_score(y_va_anom, anomaly_score))

    rows.append({"fold": fold, "model": "OneClassGraphSVM(WL)", "auc_anomaly": auc, "normal": NORMAL_LABEL, "n_train_graphs": len(G_tr), "n_val_graphs": len(G_va)})

pd.DataFrame(rows)


# Abordagem 4: Classificação binária sMCI vs pMCI

## Modelagem Tabular

In [ ]:
# Experimento E1: Elastic Net Logistic (tabular)

from sklearn.linear_model import LogisticRegression

C = 0.2
l1_ratio = 0.5

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=False)
    X_tr, y_tr = split_xy(tr)
    X_va, y_va = split_xy(va)

    common = [c for c in X_tr.columns if c in X_va.columns]
    X_tr = X_tr[common]
    X_va = X_va[common]

    clf = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=l1_ratio,
        C=C,
        max_iter=5000,
        n_jobs=-1,
        class_weight="balanced",
        random_state=RNG_SEED,
    )
    clf.fit(X_tr, y_tr)

    prob = clf.predict_proba(X_va)[:, 1]
    pred = (prob >= 0.5).astype(int)
    m = eval_binary(y_va, prob, pred)
    m.update({"fold": fold, "model": "ENetLogReg", "n_features": X_tr.shape[1]})
    rows.append(m)

pd.DataFrame(rows)


In [ ]:
# Experimento E2: Linear SVM (tabular)

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=False)
    X_tr, y_tr = split_xy(tr)
    X_va, y_va = split_xy(va)

    common = [c for c in X_tr.columns if c in X_va.columns]
    X_tr = X_tr[common]
    X_va = X_va[common]

    base = LinearSVC(C=1.0, class_weight="balanced", random_state=RNG_SEED)
    clf = CalibratedClassifierCV(base, method="sigmoid", cv=3)
    clf.fit(X_tr, y_tr)

    prob = clf.predict_proba(X_va)[:, 1]
    pred = (prob >= 0.5).astype(int)
    m = eval_binary(y_va, prob, pred)
    m.update({"fold": fold, "model": "LinearSVM", "n_features": X_tr.shape[1]})
    rows.append(m)

pd.DataFrame(rows)


In [ ]:
# Experimento E3: XGBoost (tabular) — opcional

try:
    import xgboost as xgb
except Exception as e:
    raise SystemExit("xgboost não disponível") from e

rows = []
for fold in range(N_SPLITS):
    tr, va = load_fold(fold, selected=True)  # recomenda usar após seleção
    X_tr, y_tr = split_xy(tr)
    X_va, y_va = split_xy(va)

    common = [c for c in X_tr.columns if c in X_va.columns]
    X_tr = X_tr[common]
    X_va = X_va[common]

    clf = xgb.XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.6,
        reg_lambda=1.0,
        reg_alpha=0.0,
        random_state=RNG_SEED,
        n_jobs=-1,
        eval_metric="logloss",
    )
    clf.fit(X_tr, y_tr)

    prob = clf.predict_proba(X_va)[:, 1]
    pred = (prob >= 0.5).astype(int)
    m = eval_binary(y_va, prob, pred)
    m.update({"fold": fold, "model": "XGBoost", "n_features": X_tr.shape[1]})
    rows.append(m)

pd.DataFrame(rows)


## Modelagem longitudinal

In [ ]:
# Longitudinal 2 (deltas 12/13/23) a partir do parquet wide
# Monta X_seq com shape (N, T=3, F_step) onde T=[12,13,23]

# OBS: requer pandas com suporte a parquet (pyarrow/fastparquet).

PAIR_ORDER = ["12", "13", "23"]

if not PARQUET_WIDE.is_file():
    raise SystemExit(f"Parquet não encontrado: {PARQUET_WIDE}")

wide = pd.read_parquet(PARQUET_WIDE)

# IDs e label
id_cols = [c for c in ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "GROUP"] if c in wide.columns]

# identifica colunas por pair via sufixo '__..._p12'
feat_cols = [c for c in wide.columns if "__" in c]

pair_to_cols = {}
for p in PAIR_ORDER:
    suf = f"_p{p}"
    cols = [c for c in feat_cols if c.endswith(suf)]
    pair_to_cols[p] = cols

# garante mesmo conjunto de 'features base' entre pares (pode haver pequenas diferenças)
min_cols = min(len(v) for v in pair_to_cols.values())
print({p: len(cols) for p, cols in pair_to_cols.items()}, "min=", min_cols)

# para comparabilidade: mantém a interseção de colunas dos 3 pares
common_cols = set(pair_to_cols[PAIR_ORDER[0]])
for p in PAIR_ORDER[1:]:
    common_cols &= set(pair_to_cols[p])
common_cols = sorted(common_cols)

# reordena para cada p, mantendo mesma base de features
pair_cols_ordered = {p: [c for c in common_cols if c.endswith(f"_p{p}")] for p in PAIR_ORDER}

print("common per pair", {p: len(v) for p, v in pair_cols_ordered.items()})

# Monta tensor (aqui como numpy) — você vai alimentar no PyTorch depois
X_steps = []
for p in PAIR_ORDER:
    Xp = wide[pair_cols_ordered[p]].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
    X_steps.append(Xp)

X_seq = np.stack(X_steps, axis=1)  # (N, 3, F)

y = wide["GROUP"].astype(str).map({"sMCI": 0, "pMCI": 1}).to_numpy(dtype=np.int64)

print("X_seq", X_seq.shape, "y", y.shape)


In [ ]:
# Longitudinal 1 (t1,t2,t3): monta sequência por imagem usando features_merged_combat.csv
# Produz um vetor por tempo agregando por imagem (média sobre ROIs) — baseline rápido.

FEATURES_IMG = BASE / "csvs/features_merged_combat.csv"

if not FEATURES_IMG.is_file():
    raise SystemExit(f"Arquivo ausente: {FEATURES_IMG}")

feat_img = pd.read_csv(FEATURES_IMG)
need = {"ID_IMG", "roi", "side", "label"}
if not need.issubset(set(feat_img.columns)):
    raise SystemExit(f"features_merged_combat.csv sem colunas {sorted(need - set(feat_img.columns))}")

# pega apenas colunas numéricas
meta = {"ID_IMG", "roi", "side", "label"}
num_cols = []
Z = feat_img.copy()
for c in Z.columns:
    if c in meta:
        continue
    s = Z[c]
    if pd.api.types.is_numeric_dtype(s):
        num_cols.append(c)
    else:
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().any():
            Z[c] = s_num
            num_cols.append(c)

# agrega por imagem (média sobre ROIs)
img_vec = Z.groupby("ID_IMG", as_index=True)[num_cols].mean()
print("n_images", img_vec.shape[0], "F", img_vec.shape[1])

# Exemplo: montar sequência para um fold (repita por fold para CV-safe)
fold = 0
tr, va = load_fold(fold, selected=False)

# conjuntos únicos no fold
keys = ["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX", "GROUP", "ID_IMG_i1", "ID_IMG_i2", "ID_IMG_i3"]
tr_sets = tr[keys].drop_duplicates(subset=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]).copy()
va_sets = va[keys].drop_duplicates(subset=["ID_PT", "COMBINATION_NUMBER", "TRIPLET_IDX"]).copy()


def build_seq(df_sets: pd.DataFrame):
    ids1 = df_sets["ID_IMG_i1"].astype(str)
    ids2 = df_sets["ID_IMG_i2"].astype(str)
    ids3 = df_sets["ID_IMG_i3"].astype(str)

    X1 = img_vec.reindex(ids1).to_numpy(dtype=np.float32)
    X2 = img_vec.reindex(ids2).to_numpy(dtype=np.float32)
    X3 = img_vec.reindex(ids3).to_numpy(dtype=np.float32)

    X = np.stack([X1, X2, X3], axis=1)  # (N,3,F)
    y = df_sets["GROUP"].astype(str).map({"sMCI": 0, "pMCI": 1}).to_numpy(dtype=np.int64)
    return X, y

X_tr_seq, y_tr = build_seq(tr_sets)
X_va_seq, y_va = build_seq(va_sets)

print("train", X_tr_seq.shape, "val", X_va_seq.shape)
print("missing rows train", np.isnan(X_tr_seq).any(axis=(1,2)).sum(), "val", np.isnan(X_va_seq).any(axis=(1,2)).sum())


## Modelagem Grafos

In [ ]:
# Grafos inter-tempo (esqueleto): construir dados para PyTorch Geometric
# Nós: roi_side (20)
# Features por nó: exemplo usando deltas do parquet wide (p12/p13/p23)

# Este bloco só prepara tensores; o treinamento GNN (GCN/GraphSAGE/GAT) vem depois.

# 1) Define a ordem de nós (roi_side) a partir do features_all do seu parquet (recomendado fixar via ROI_TABLE)
# Aqui: extrai dos nomes de colunas do parquet wide.

feat_cols = [c for c in wide.columns if "__" in c]

# recupera sufixos únicos "roi_side_pair" dos nomes das colunas
suffixes = sorted({c.split("__",1)[1] for c in feat_cols})
# reduz a roi_side (removendo _p12/_p13/_p23)
roi_side = sorted({s.rsplit("_p",1)[0] for s in suffixes})
print("n_roi_side", len(roi_side))

# 2) para cada roi_side, pegue features de cada pair e concatene no nó
# Ex.: node_feat = [all_features_p12, all_features_p13, all_features_p23]

# identifica a base de feature (prefixo antes de '__')
base_feats = sorted({c.split("__",1)[0] for c in feat_cols})

# para manter tamanho manejável, você provavelmente vai selecionar um subconjunto (via seleção por fold)
print("base_feats", len(base_feats))

# exemplo: construir matriz de nós para 1 amostra (índice 0)
idx = 0
node_features = []
for rs in roi_side:
    vec_parts = []
    for p in PAIR_ORDER:
        cols = [f"{bf}__{rs}_p{p}" for bf in base_feats if f"{bf}__{rs}_p{p}" in wide.columns]
        x = wide.loc[idx, cols].to_numpy(dtype=np.float32)
        vec_parts.append(x)
    node_features.append(np.concatenate(vec_parts, axis=0))

X_nodes = np.stack(node_features, axis=0)  # (n_nodes, F_node)
print("X_nodes", X_nodes.shape)

# 3) arestas (edge_index) — comece com um grafo fixo simples (ex.: fully-connected sem self-loop) ou um atlas
# Ex.: fully connected (custo O(N^2), mas N=20 é pequeno)

edges = []
for i in range(len(roi_side)):
    for j in range(len(roi_side)):
        if i == j:
            continue
        edges.append([i, j])
edge_index = np.array(edges, dtype=np.int64).T  # (2, E)
print("edge_index", edge_index.shape)

# 4) y
print("y", int(wide.loc[idx, "GROUP"]) if "GROUP" in wide.columns else None)
